# Run TMB_RFD2 inference

This notebook is a beginner-friendly example for running **TMB_RFD2** after you have:

1. cloned the `TMB_pipeline` repository
2. downloaded the RFD2 checkpoint
3. downloaded the Apptainer/Singularity image
4. set the environment variables from the README

The notebook avoids personal machine-specific paths and instead uses environment variables:

- `RFD2_CKPT_PATH`
- `APPTAINER_PATH`
- optional: `TMB_PIPELINE_DIR`

It runs one example condition first, which is the safest way to test that everything is working.


## 1. Imports


In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

## 2. Read environment variables

Make sure you already ran the setup commands from the README in your shell before starting Jupyter.


In [ ]:
RFD2_CKPT_PATH = os.environ.get("RFD2_CKPT_PATH")
APPTAINER_PATH = os.environ.get("APPTAINER_PATH")
TMB_PIPELINE_DIR = os.environ.get("TMB_PIPELINE_DIR", os.getcwd())

assert RFD2_CKPT_PATH, "RFD2_CKPT_PATH is not set. Follow the README setup first."
assert APPTAINER_PATH, "APPTAINER_PATH is not set. Follow the README setup first."

TMB_PIPELINE_DIR = Path(TMB_PIPELINE_DIR).resolve()
RFD2_CKPT_PATH = Path(RFD2_CKPT_PATH).resolve()
APPTAINER_PATH = Path(APPTAINER_PATH).resolve()

print("TMB_PIPELINE_DIR =", TMB_PIPELINE_DIR)
print("RFD2_CKPT_PATH   =", RFD2_CKPT_PATH)
print("APPTAINER_PATH   =", APPTAINER_PATH)


## 3. User-editable settings

Edit these if your repo structure or example input locations are different.


In [ ]:
# -----------------------------
# USER SETTINGS
# -----------------------------

# Repository code locations
RFD2_CODE_DIR = TMB_PIPELINE_DIR / "TMB_RFD2"
ADJ_SS_WRAPPER_SCRIPT = TMB_PIPELINE_DIR / "scripts" / "adj_ss_wrapper_glykink.py"

# Update this to a real example PDB inside your repo
INPUT_PDB = TMB_PIPELINE_DIR / "example_inputs" / "input.pdb"

# RFD2 settings
CONFIG_NAME = "aa_ppi"
HAS_TERMINI = "[True]"
CONTIGS = "['1-1']"
CENTER_TYPE = "all"
NUM_DESIGNS = 10

# Example design condition
N_STRANDS = 12
RESIDUES_PER_STRAND = 16
GLYCINE_TWISTS = "1 2 7 8"

# Output locations
WORK_DIR = TMB_PIPELINE_DIR / "notebook_runs"
ADJ_SS_DIR = WORK_DIR / "adj_ss"
OUTPUT_DIR = WORK_DIR / "backbones"
COMMANDS_FILE = WORK_DIR / "inference_commands.txt"

WORK_DIR.mkdir(parents=True, exist_ok=True)
ADJ_SS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Work directory:", WORK_DIR)


## 4. Check that required files exist


In [ ]:
assert RFD2_CODE_DIR.exists(), f"Could not find RFD2 code directory: {RFD2_CODE_DIR}"
assert ADJ_SS_WRAPPER_SCRIPT.exists(), f"Could not find wrapper script: {ADJ_SS_WRAPPER_SCRIPT}"
assert RFD2_CKPT_PATH.exists(), f"Checkpoint does not exist: {RFD2_CKPT_PATH}"
assert APPTAINER_PATH.exists(), f"Apptainer image does not exist: {APPTAINER_PATH}"

if not INPUT_PDB.exists():
    print(f"Warning: input PDB not found at {INPUT_PDB}")
    print("Update INPUT_PDB in the settings cell before running inference.")
else:
    print("Input PDB:", INPUT_PDB)


## 5. Helper functions


In [ ]:
def run_command(cmd, cwd=None, env=None, check=True):
    print("\nRunning command:\n")
    if isinstance(cmd, list):
        print(" ".join(shlex.quote(str(x)) for x in cmd))
    else:
        print(cmd)

    result = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
        shell=isinstance(cmd, str),
    )

    if result.stdout:
        print("\nSTDOUT:\n")
        print(result.stdout)

    if result.stderr:
        print("\nSTDERR:\n")
        print(result.stderr)

    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")

    return result


def build_twist_residues(glycine_twists: str, residues_per_strand: int):
    twist_residues = []
    for strand_str in glycine_twists.split():
        strand = int(strand_str)
        start = (strand - 1) * residues_per_strand + residues_per_strand // 2 - 2
        for offset in range(5):
            twist_residues.append(start + offset)
    return twist_residues


## 6. Generate adjacency / secondary-structure / twist tensors


In [ ]:
twist_residues = build_twist_residues(GLYCINE_TWISTS, RESIDUES_PER_STRAND)
glycine_twists_res = ",".join(map(str, twist_residues))

this_adj_ss_dir = ADJ_SS_DIR / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_glycine_twists_{GLYCINE_TWISTS.replace(' ', '_')}"
this_adj_ss_dir.mkdir(parents=True, exist_ok=True)

adj_file = this_adj_ss_dir / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_adj.pt"
ss_file = this_adj_ss_dir / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_ss.pt"
twist_file = this_adj_ss_dir / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_twist.pt"

if adj_file.exists() and ss_file.exists() and twist_file.exists():
    print("Adjacency / SS / twist files already exist. Skipping generation.")
else:
    cmd = [
        "python",
        str(ADJ_SS_WRAPPER_SCRIPT),
        "--n", str(N_STRANDS),
        "--residues_per_strand", str(RESIDUES_PER_STRAND),
        "--twist_indices", *GLYCINE_TWISTS.split(),
        "--output_dir", str(this_adj_ss_dir),
    ]
    run_command(cmd, cwd=TMB_PIPELINE_DIR)

print("Adjacency dir:", this_adj_ss_dir)
print("Twist residues:", glycine_twists_res)


## 7. Build the Apptainer command for RFD2 inference


In [ ]:
run_name = f"glycine_twists_n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_{GLYCINE_TWISTS.replace(' ', '_')}"
this_output_dir = OUTPUT_DIR / run_name
this_output_dir.mkdir(parents=True, exist_ok=True)

output_prefix = this_output_dir / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}"
pythonpath = str(TMB_PIPELINE_DIR) + ":" + os.environ.get("PYTHONPATH", "")

inner_cmd = f"""
cd {shlex.quote(str(TMB_PIPELINE_DIR))}
export PYTHONPATH={shlex.quote(pythonpath)}

python {shlex.quote(str(RFD2_CODE_DIR / 'rf_diffusion' / 'run_inference.py'))} \
  --config-name={shlex.quote(CONFIG_NAME)} \
  contigmap.has_termini={shlex.quote(HAS_TERMINI)} \
  inference.output_prefix={shlex.quote(str(output_prefix))} \
  scaffoldguided.scaffold_dir={shlex.quote(str(this_adj_ss_dir))} \
  contigmap.contigs={shlex.quote(CONTIGS)} \
  inference.ckpt_path={shlex.quote(str(RFD2_CKPT_PATH))} \
  inference.input_pdb={shlex.quote(str(INPUT_PDB))} \
  inference.num_designs={shlex.quote(str(NUM_DESIGNS))} \
  ++transforms.configs.CenterPostTransform.center_type={shlex.quote(CENTER_TYPE)} \
  ++glycine_twists.glycine_twists_res={shlex.quote(glycine_twists_res)}
""".strip()

apptainer_cmd = [
    "apptainer", "exec", "--nv",
    str(APPTAINER_PATH),
    "bash", "-lc", inner_cmd
]

print("Ready to run inference.")
print("Output prefix:", output_prefix)


## 8. Save the command to a text file


In [ ]:
with open(COMMANDS_FILE, "a") as f:
    f.write(" ".join(shlex.quote(str(x)) for x in apptainer_cmd) + "\n")

print("Saved command to:", COMMANDS_FILE)


## 9. Run inference

Run this cell only after the previous cells complete successfully.


In [ ]:
run_command(apptainer_cmd, cwd=TMB_PIPELINE_DIR)


## 10. Inspect outputs


In [ ]:
for path in sorted(this_output_dir.rglob("*")):
    print(path)


## Optional: run a small batch of twist combinations

Only use this after the single example above works.


In [ ]:
twist_combinations = [
    "1 2 4 5",
    "1 2 5 6",
    "1 2 6 7",
]

pythonpath = str(TMB_PIPELINE_DIR) + ":" + os.environ.get("PYTHONPATH", "")

for glycine_twists in twist_combinations:
    print("\n==============================")
    print("Running:", glycine_twists)
    print("==============================")

    twist_residues = build_twist_residues(glycine_twists, RESIDUES_PER_STRAND)
    glycine_twists_res = ",".join(map(str, twist_residues))

    this_adj_ss_dir = ADJ_SS_DIR / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_glycine_twists_{glycine_twists.replace(' ', '_')}"
    this_adj_ss_dir.mkdir(parents=True, exist_ok=True)

    adj_file = this_adj_ss_dir / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_adj.pt"
    ss_file = this_adj_ss_dir / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_ss.pt"
    twist_file = this_adj_ss_dir / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_twist.pt"

    if not (adj_file.exists() and ss_file.exists() and twist_file.exists()):
        cmd = [
            "python",
            str(ADJ_SS_WRAPPER_SCRIPT),
            "--n", str(N_STRANDS),
            "--residues_per_strand", str(RESIDUES_PER_STRAND),
            "--twist_indices", *glycine_twists.split(),
            "--output_dir", str(this_adj_ss_dir),
        ]
        run_command(cmd, cwd=TMB_PIPELINE_DIR)

    run_name = f"glycine_twists_n{N_STRANDS}_nres{RESIDUES_PER_STRAND}_{glycine_twists.replace(' ', '_')}"
    this_output_dir = OUTPUT_DIR / run_name
    this_output_dir.mkdir(parents=True, exist_ok=True)
    output_prefix = this_output_dir / f"n{N_STRANDS}_nres{RESIDUES_PER_STRAND}"

    inner_cmd = f"""
cd {shlex.quote(str(TMB_PIPELINE_DIR))}
export PYTHONPATH={shlex.quote(pythonpath)}

python {shlex.quote(str(RFD2_CODE_DIR / 'rf_diffusion' / 'run_inference.py'))} \
  --config-name={shlex.quote(CONFIG_NAME)} \
  contigmap.has_termini={shlex.quote(HAS_TERMINI)} \
  inference.output_prefix={shlex.quote(str(output_prefix))} \
  scaffoldguided.scaffold_dir={shlex.quote(str(this_adj_ss_dir))} \
  contigmap.contigs={shlex.quote(CONTIGS)} \
  inference.ckpt_path={shlex.quote(str(RFD2_CKPT_PATH))} \
  inference.input_pdb={shlex.quote(str(INPUT_PDB))} \
  inference.num_designs={shlex.quote(str(NUM_DESIGNS))} \
  ++transforms.configs.CenterPostTransform.center_type={shlex.quote(CENTER_TYPE)} \
  ++glycine_twists.glycine_twists_res={shlex.quote(glycine_twists_res)}
""".strip()

    apptainer_cmd = [
        "apptainer", "exec", "--nv",
        str(APPTAINER_PATH),
        "bash", "-lc", inner_cmd
    ]

    with open(COMMANDS_FILE, "a") as f:
        f.write(" ".join(shlex.quote(str(x)) for x in apptainer_cmd) + "\n")

    run_command(apptainer_cmd, cwd=TMB_PIPELINE_DIR)


In [ ]:
# Run multiple strand-number / residues-per-strand settings

strand_residue_map = {
    10: [16],
    12: [16],
    14: [16],
    16: [16],
}

glycine_twist_map = {
    10: ["1 3", "1 4", "1 5"],
    12: ["1 3", "1 4", "1 5"],
    14: ["1 3", "1 4", "1 5"],
    16: ["1 3", "1 4", "1 5"],
}

pythonpath = str(TMB_PIPELINE_DIR) + ":" + os.environ.get("PYTHONPATH", "")

for n_strands, residues_per_strand_list in strand_residue_map.items():
    for residues_per_strand in residues_per_strand_list:
        for glycine_twists in glycine_twist_map[n_strands]:
            print("\n==============================")
            print(f"Running n={n_strands}, nres={residues_per_strand}, twists={glycine_twists}")
            print("==============================")

            twist_residues = build_twist_residues(glycine_twists, residues_per_strand)
            glycine_twists_res = ",".join(map(str, twist_residues))

            this_adj_ss_dir = ADJ_SS_DIR / (
                f"n{n_strands}_nres{residues_per_strand}_glycine_twists_{glycine_twists.replace(' ', '_')}"
            )
            this_adj_ss_dir.mkdir(parents=True, exist_ok=True)

            adj_file = this_adj_ss_dir / f"n{n_strands}_nres{residues_per_strand}_adj.pt"
            ss_file = this_adj_ss_dir / f"n{n_strands}_nres{residues_per_strand}_ss.pt"
            twist_file = this_adj_ss_dir / f"n{n_strands}_nres{residues_per_strand}_twist.pt"

            if not (adj_file.exists() and ss_file.exists() and twist_file.exists()):
                cmd = [
                    "python",
                    str(ADJ_SS_WRAPPER_SCRIPT),
                    "--n", str(n_strands),
                    "--residues_per_strand", str(residues_per_strand),
                    "--twist_indices", *glycine_twists.split(),
                    "--output_dir", str(this_adj_ss_dir),
                ]
                run_command(cmd, cwd=TMB_PIPELINE_DIR)

            run_name = (
                f"glycine_twists_n{n_strands}_nres{residues_per_strand}_{glycine_twists.replace(' ', '_')}"
            )
            this_output_dir = OUTPUT_DIR / run_name
            this_output_dir.mkdir(parents=True, exist_ok=True)
            output_prefix = this_output_dir / f"n{n_strands}_nres{residues_per_strand}"

            inner_cmd = f"""
cd {shlex.quote(str(TMB_PIPELINE_DIR))}
export PYTHONPATH={shlex.quote(pythonpath)}

python {shlex.quote(str(RFD2_CODE_DIR / 'rf_diffusion' / 'run_inference.py'))} \\
  --config-name={shlex.quote(CONFIG_NAME)} \\
  contigmap.has_termini={shlex.quote(HAS_TERMINI)} \\
  inference.output_prefix={shlex.quote(str(output_prefix))} \\
  scaffoldguided.scaffold_dir={shlex.quote(str(this_adj_ss_dir))} \\
  contigmap.contigs={shlex.quote(CONTIGS)} \\
  inference.ckpt_path={shlex.quote(str(RFD2_CKPT_PATH))} \\
  inference.input_pdb={shlex.quote(str(INPUT_PDB))} \\
  inference.num_designs={shlex.quote(str(NUM_DESIGNS))} \\
  ++transforms.configs.CenterPostTransform.center_type={shlex.quote(CENTER_TYPE)} \\
  ++glycine_twists.glycine_twists_res={shlex.quote(glycine_twists_res)}
""".strip()

            apptainer_cmd = [
                "apptainer", "exec", "--nv",
                str(APPTAINER_PATH),
                "bash", "-lc", inner_cmd
            ]

            with open(COMMANDS_FILE, "a") as f:
                f.write(" ".join(shlex.quote(str(x)) for x in apptainer_cmd) + "\n")

            run_command(apptainer_cmd, cwd=TMB_PIPELINE_DIR)